In [2]:
%pip install findspark

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [1]:
import findspark
findspark.init()
findspark.find()

'/usr/lib/spark'

In [2]:
from pyspark.sql import SparkSession

In [3]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SQLContext

In [4]:
!hdfs dfs -ls

zsh:1: command not found: hdfs


In [5]:
!hdfs dfs -ls /tmp

Found 1 items
drwxrwxrwt   - hdfs hadoop          0 2025-07-22 18:28 /tmp/hadoop-yarn


In [4]:
!pwd

/home/ubuntu/ml


In [6]:
!hdfs dfs -put ./abalone_train.csv /tmp

In [7]:
!hdfs dfs -ls /tmp

Found 2 items
-rw-r--r--   1 ubuntu hadoop     145611 2025-07-22 18:45 /tmp/abalone_train.csv
drwxrwxrwt   - hdfs   hadoop          0 2025-07-22 18:28 /tmp/hadoop-yarn


In [9]:
spark = SparkSession\
    .builder\
    .appName("mytestapp")\
    .getOrCreate()

In [16]:
sql = SQLContext(spark)

In [17]:
sql

In [ ]:
df = sql.read.parquet("s3a://yc-mdb-examples/dataproc/example01/set01")

In [19]:
defaultFS = spark._jsc.hadoopConfiguration().get("fs.defaultFS")
month_stat = df.groupBy("Month").count()
month_stat.repartition(1).write.format("csv").save(defaultFS+"/tmp/month_stat2")

In [33]:
!hdfs dfs -ls /tmp/

Found 3 items
-rw-r--r--   1 ubuntu hadoop     145611 2025-07-22 15:31 /tmp/abalone_train.csv
drwxrwxrwt   - hdfs   hadoop          0 2025-07-22 14:59 /tmp/hadoop-yarn
drwxr-xr-x   - ubuntu hadoop          0 2025-07-22 15:34 /tmp/month_stat2


In [30]:
!hdfs dfs -ls /tmp/month_stat2

Found 2 items
-rw-r--r--   1 ubuntu hadoop          0 2025-07-22 15:34 /tmp/month_stat2/_SUCCESS
-rw-r--r--   1 ubuntu hadoop        111 2025-07-22 15:34 /tmp/month_stat2/part-00000-a7a45e66-0f81-44cb-9157-14251a05a7a0-c000.csv


In [31]:
m = spark.read.option('header', 'true').csv('/tmp/month_stat2/part-00000-a7a45e66-0f81-44cb-9157-14251a05a7a0-c000.csv')

In [32]:
m.show(5)

+---+------+
|  7|645299|
+---+------+
|  6|626193|
|  9|585749|
|  5|616529|
|  1|570118|
| 10|616101|
+---+------+
only showing top 5 rows



In [23]:
!pwd

/home/ubuntu/ml


In [15]:
data = (spark.read.format('csv')
        .options(header='false', inferSchema='true')
        .load('/tmp/abalone_train.csv'))

In [16]:
type(data)

pyspark.sql.dataframe.DataFrame

In [17]:
data.schema

StructType(List(StructField(_c0,DoubleType,true),StructField(_c1,DoubleType,true),StructField(_c2,DoubleType,true),StructField(_c3,DoubleType,true),StructField(_c4,DoubleType,true),StructField(_c5,DoubleType,true),StructField(_c6,DoubleType,true),StructField(_c7,IntegerType,true)))

In [18]:
data.printSchema()

root
 |-- _c0: double (nullable = true)
 |-- _c1: double (nullable = true)
 |-- _c2: double (nullable = true)
 |-- _c3: double (nullable = true)
 |-- _c4: double (nullable = true)
 |-- _c5: double (nullable = true)
 |-- _c6: double (nullable = true)
 |-- _c7: integer (nullable = true)



In [19]:
for r in data.dtypes:
    print(f"{r[0]:>10}\t{r[1]}")

       _c0	double
       _c1	double
       _c2	double
       _c3	double
       _c4	double
       _c5	double
       _c6	double
       _c7	int


In [20]:
data.show(5)

+-----+-----+-----+-----+------+------+------+---+
|  _c0|  _c1|  _c2|  _c3|   _c4|   _c5|   _c6|_c7|
+-----+-----+-----+-----+------+------+------+---+
|0.435|0.335| 0.11|0.334|0.1355|0.0775|0.0965|  7|
|0.655| 0.51| 0.16|1.092| 0.396|0.2825|  0.37| 14|
|0.545|0.425|0.125|0.768| 0.294|0.1495|  0.26| 16|
|0.625|0.495|0.155|1.025|  0.46|0.1945|  0.34|  9|
|0.615|0.495| 0.16|1.255|0.5815|0.3195|0.3225| 12|
+-----+-----+-----+-----+------+------+------+---+
only showing top 5 rows



In [21]:
spark.stop()